#1.Introduction

# Retail Sales Performance Analysis

### Project Overview

This project analyzes retail sales data to uncover business insights related to sales performance, profitability, customer behavior, product performance, and regional trends. The analysis includes data cleaning, feature engineering, exploratory data analysis (EDA), data visualization, and business recommendations to support data-driven decision-making.

# 2.Project Objectives

- Understand overall sales performance.
- Identify the most profitable product categories.
- Analyze customer purchasing behavior.
- Identify loss-making products and orders.
- Study the impact of discounts on profitability.
- Analyze regional and state-wise sales performance.
- Generate business insights and actionable recommendations.

# 3.Importing Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick

# 4.Load Dataset

In [ ]:
data = pd.read_csv("/content/Sample - Superstore.csv", encoding = 'latin1')
df = pd.DataFrame(data)

# 5.Data Understanding

In [ ]:
print("=" * 80)
print(f"Total Rows And Columns :")
print("=" * 80)
print(df.shape)
print("=" * 80)
print(f"Column Names :")
print("=" * 80)
print(df.columns)
print("=" * 80)
print(f"Random Rows :")
print("=" * 80)
print(df.sample(5))
print("=" * 80)
print(f"Data Full Information :")
print("=" * 80)
print(df.info())
print("=" * 80)
print(f"Empty Values :")
print("=" * 80)
print(df.isnull().sum())
print("=" * 80)
print(f"Duplicate Values :")
print("=" * 80)
print(df.duplicated().sum())
print("=" * 80)
print(f"Statictics :")
print("=" * 80)
print(df.describe())

# 6.Feature Engineering

**1.Month wise Order Column**

In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Order Month"] = df["Order Date"].dt.to_period("M")

monthly_sales = (
    df.groupby("Order Month")["Sales"]
    .sum()
    .round()
)

print(monthly_sales)


**2.After Discount order Price Column**

In [ ]:
df["Discounted Price"] = (df["Sales"] * (1 - df["Discount"])).round()

print("=" * 20)
print("Discounted Price")
print("=" * 20)
print(df["Discounted Price"].sample(5))

# 7.Exploratory Data Analysis (EDA)

## Function to group and sum values.

In [ ]:
def top_n(df, groupby_col, val_col, n=10, ascending=False):
  return(
      (df.groupby(groupby_col)[val_col]
      .sum()
      .sort_values(ascending=ascending)
      .head(n))
      .round(2)
  )

**# 1.Category Analysis**

In [ ]:
category_wise_summary = df.groupby("Category").agg({
    "Sales" : "sum",
    "Profit" : "sum",
}).round(2)

print("=" * 50)
print("Category-wise Sales and Profit")
print("=" * 50)
print(category_wise_summary)

overall_profit_margin = (
    (df.groupby("Category")["Profit"].sum() /
     df.groupby("Category")["Sales"].sum())
    .mul(100)
    .round(2)
)

print("=" * 35)
print("Overall profit margin by category")
print("=" * 35)
print(overall_profit_margin)

**# 2.Product Analysis**

In [ ]:
print("=" * 90)
print("Top 10 Products by Sales")
print("=" * 90)
print(top_n(df, "Product Name", "Sales"))

print("=" * 90)
print("Top 10 Profit-Making Products")
print("=" * 90)
print(top_n(df, "Product Name", "Profit"))

print("=" * 90)
print("Bottom 10 Loss-Making Products")
print("=" * 90)
print(top_n(df, "Product Name", "Profit", ascending=True))

**# 3.Customer Analysis**

In [ ]:
print("=" * 90)
print("Top 10 Customers by Sales")
print("=" * 90)
print(top_n(df, "Customer Name", "Sales"))

print("=" * 90)
print("Top 10 Profit-Making Customers")
print("=" * 90)
print(top_n(df, "Customer Name", "Profit"))

print("=" * 90)
print("Bottom 10 Loss-Making Customers")
print("=" * 90)
print(top_n(df, "Customer Name", "Profit", ascending=True))


**4.State Analysis**

In [ ]:
print("=" * 90)
print("Top 10 States by Sales")
print("=" * 90)
print(top_n(df, "State", "Sales"))

print("=" * 90)
print("Top 10 Profit-Making States")
print("=" * 90)
print(top_n(df, "State", "Profit"))

print("=" * 90)
print("Bottom 10 Loss-Making States")
print("=" * 90)
print(top_n(df, "State", "Profit", ascending=True))


state_summary = df.groupby("State").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
).round(2)

high_sales_neg_profit_states = state_summary[
    (state_summary["Total_Sales"] > 0) &
    (state_summary["Total_Profit"] < 0)
].sort_values(by="Total_Profit", ascending=True).head(10)
print("=" * 90)
print("States with High Sales but Negative Profit")
print("=" * 90)
print(high_sales_neg_profit_states)

**5.Monthly Sales**

In [ ]:
print("=" * 90)
print("Top 10 Monthls by Sales")
print("=" * 90)
print(top_n(df, "Order Month", "Sales"))

print("=" * 90)
print("Top 10 Months by Profit")
print("=" * 90)
print(top_n(df, "Order Month", "Profit"))

**6.Discount Analysis**

In [ ]:
filtered_orders = (df[(df["Profit"] < 0) & (df["Discount"] > 0.3)]).round(2)


print("=" * 55)
print("loss making Orders because of high discounts")
print("=" * 55)
print(filtered_orders[["Order ID", "Customer Name", "Discount", "Profit", "Product Name"]])


**7.Region Analysis**

In [ ]:
print("=" * 80)
print("Region wise Sales")
print("=" * 80)
print(top_n(df, "Region", "Sales"))

print("=" * 80)
print("Region wise Profits")
print("=" * 80)
print(top_n(df, "Region", "Profit"))


# 8.Visualizations

**1.Top 10 Products generates the highest Profit Horizontal Bar**

In [ ]:
top_10_products = (top_n(df, "Product Name", "Profit"))
print(top_10_products)

plt.figure(figsize=(10, 6))
plt.barh(top_10_products.index, top_10_products.values,color="green")
plt.title("Top 10 most profitable products")
plt.xlabel("Profit")
plt.ylabel("Product Name")
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()
plt.savefig("top_10_most_profitable_products.png")


**2.Bottom 10 products are causing the biggest losses Horizontal Bar**

In [ ]:
top_10_losses = (top_n(df, "Product Name", "Profit", ascending=True))

print(top_10_losses)

plt.figure(figsize=(10,6))
plt.barh(top_10_losses.index, top_10_losses.values,color="red")
plt.title("Top 10 loss-making Products")
plt.xlabel("Profit")
plt.ylabel("Product Name")
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()
plt.savefig("top_10_loss_making_products.png")


**3.Monthly Sales Trend Line Chart**

In [ ]:
monthly_sales = df.groupby("Order Month")["Sales"].sum().round()

plt.figure(figsize=(12,6))
plt.plot(
    monthly_sales.index.astype(str),
    monthly_sales.values,
    marker="o",
    color="orange"
)
plt.title("Monthly Sales Trend")
plt.xlabel("Order Month")
plt.ylabel("Sales")
plt.tight_layout()
plt.grid(alpha=0.5)
plt.xticks(rotation=45)
plt.gca().yaxis.set_major_formatter(
    mtick.StrMethodFormatter('{x:,.0f}')
)
plt.show()
plt.savefig("monthly_sales_trend.png")


**4.Two numeric variables → Scatter Plot**

In [ ]:
plt.figure(figsize=(12,6))
plt.scatter(
    df["Discount"],
    df["Profit"],
    color="brown"
)
plt.title("Discount vs Profit")
plt.xlabel("Discout")
plt.ylabel("Profit")
plt.grid(alpha=0.5)
plt.tight_layout()
plt.show()
plt.savefig("discount_vs_profit.png")

**5.Which states generate the highest sales and profit Bar Graph**

In [ ]:
state_sales_profit = (
    df.groupby("State")
    [["Sales", "Profit"]].sum()
    .sort_values(by="Profit", ascending=False)
    .head(10)
    .round()
)

plt.figure(figsize=(12,6))
plt.bar(
    state_sales_profit.index,
    state_sales_profit["Profit"],
    color="green"
)
plt.title("Sales and Profit by State")
plt.xlabel("Sates")
plt.ylabel("Sales and Profit")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.savefig("sales_profit_by_state.png")

**7.Region-wise sales Bar Chart**

In [ ]:
region_summary = (top_n(df, "Region", "Sales"))

plt.figure(figsize=(12,6))
plt.bar(region_summary.index, region_summary.values, color="yellow")
plt.title("Region wise Sales report")
plt.xlabel("Region")
plt.ylabel("Sales")
plt.grid(alpha=0.5)
plt.tight_layout()
plt.show()
plt.savefig("region_wise_sales.png")

**7.Correlation Heatmap**

In [ ]:
correlation_df = (
    df.select_dtypes(include="number")
    .drop(columns=["Row ID", "Postal Code"])
)
corr = correlation_df.corr()
print(corr)

plt.figure(figsize=(8,6))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5,
    square=True
)
plt.title("Correlation Heatmap")
plt.show()
plt.savefig("correlation_heatmap.png")

# 9.Executive KPI Summary

In [ ]:
total_sales = df["Sales"].sum().round(2)

total_profit = df["Profit"].sum().round(2)

total_orders = df["Order ID"].nunique()

total_customers = df["Customer Name"].nunique()

total_products = df["Product Name"].nunique()

avg_discount = (df["Discount"].mean()*100).round()

overall_profit_margin = ((total_profit/total_sales)*100).round()


print("=" * 50)
print("Executive KPI Summary")
print("=" * 50)

print(f"Total Sales           : ${total_sales:,.2f}")
print(f"Total Profit          : ${total_profit:,.2f}")
print(f"Total Orders          : {total_orders}")
print(f"Total Customers       : {total_customers}")
print(f"Total Products        : {total_products}")
print(f"Average Discount      : {avg_discount}%")
print(f"Overall Profit Margin : {overall_profit_margin}%")

# 10.Key Business Insights

### 1. Technology was the most profitable category.

Technology generated the highest overall profit margin, making it the strongest-performing product category.

---

### 2. Furniture requires attention.

Although Furniture generated high sales, its profit margin was significantly lower than other categories, indicating pricing or discount challenges.

---

### 3. Discounts affected profitability.

Orders with discounts greater than 30% frequently resulted in financial losses, suggesting that aggressive discounting reduced profitability.

---

### 4. California was the top-performing state.

California contributed the highest overall profit, making it one of the company's strongest markets.

---

### 5. A small number of products generated major losses.

Certain products consistently produced negative profits and should be reviewed for pricing, supplier cost, or discontinuation.

---

### 6. Customer concentration.

A relatively small group of customers generated a significant portion of total sales, highlighting the importance of customer retention.

---

### 7. Monthly sales fluctuated throughout the year.

Seasonal demand patterns indicate opportunities for inventory planning and promotional campaigns.

---

### 8. Correlation analysis.

The correlation matrix showed a negative relationship between Discount and Profit, reinforcing the impact of excessive discounting on business performance.

# 11.Business Recommendations

Based on the analysis, the following recommendations are suggested:

1. Review discount strategies for products with consistently negative profits to improve profitability.

2. Focus marketing and inventory investments on high-performing Technology products.

3. Reassess pricing, supplier costs, and discount policies for Furniture products to improve profit margins.

4. Strengthen customer retention programs for repeat customers through loyalty rewards and personalized offers.

5. Increase business expansion efforts in high-performing states such as California and New York while identifying growth opportunities in lower-performing regions.

6. Monitor loss-making products regularly and consider repricing or discontinuing products with consistently negative returns.

7. Utilize seasonal sales trends for better inventory planning, demand forecasting, and promotional campaigns.

8. Build interactive dashboards to continuously monitor key business KPIs and support management decision-making.

# 12.Conclusion

This project presented an end-to-end retail sales performance analysis using Python and data analytics techniques.

The workflow included data cleaning, feature engineering, exploratory data analysis, KPI reporting, business visualization, and interpretation of results.

The analysis identified Technology as the highest-performing category, while Furniture showed opportunities for improving profitability. High discount levels were associated with lower profits, emphasizing the importance of balanced pricing strategies.

Overall, the findings provide actionable insights that can help management improve profitability, optimize inventory, strengthen customer retention, and support data-driven business decisions.

# 13.Future Enhancements

Future versions of this project can include:

- Development of an interactive Power BI dashboard.
- Automated data refresh using APIs or scheduled ETL pipelines.
- Sales forecasting using Machine Learning models.
- Customer segmentation using clustering techniques.
- Integration with SQL databases instead of CSV files.
- Deployment as an interactive dashboard using Streamlit.

# 14.Skills Demonstrated

### Programming
- Python

### Libraries
- Pandas
- Matplotlib
- Seaborn

### Data Analysis
- Data Cleaning
- Feature Engineering
- Exploratory Data Analysis (EDA)
- KPI Reporting
- Correlation Analysis

### Business Intelligence
- Business Insights
- Business Recommendations
- Retail Sales Analytics

### Data Visualization
- Bar Charts
- Line Charts
- Scatter Plots
- Heatmaps

# 14.Exporting file in csv formata.

In [ ]:
Postal = df["Postal Code"].value().sum()
print(Postal)

AttributeError: 'int' object has no attribute 'sum'

In [ ]:
df.to_csv("Retail Sales Performance Analysis.csv", index=False)